In [ ]:
import xarray as xr
import rioxarray
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.base import RegressorMixin
from sklearn.metrics import mean_squared_error, explained_variance_score, r2_score, roc_auc_score
from sklego.meta import ZeroInflatedRegressor
from sklearn.compose import TransformedTargetRegressor
from scipy.special import logit, expit # expit == invlogit
from scipy.stats import spearmanr, pearsonr
import os
import lightgbm as lgb
import warnings
import logging
import geopandas as gpd
from itertools import product
from tqdm.autonotebook import tqdm

import const
import gbm

## Load data

In [ ]:
westmort = xr.open_zarr("../data_working/westmort.zarr/").compute().rio.write_crs(const.PROJECTION)
westmort

In [ ]:
mort_vars = list(filter(lambda x: x.endswith("target"), westmort.variables))
ba_vars   = list(filter(lambda x: x.endswith("ba"), westmort.variables))
damage = westmort[mort_vars].to_dataarray(dim="agent").assign_coords(agent=list(map(lambda x: x.replace("_target", ""), mort_vars)))
basal_area = westmort[ba_vars].to_dataarray(dim="agent").assign_coords(agent=list(map(lambda x: x.replace("_ba", ""), ba_vars)))
damage = damage.where(basal_area > 0)

In [ ]:
# Count number of agents with nonzero damage
n_nonzero = (damage > 0).sum(dim="agent")
nnz, count = np.unique(n_nonzero, return_counts=True)

for n, c in zip(nnz, count):
    print(f"{n:>2}: {c:>8}")

## Identify years with greatest mortality

For exploratory analysis, we should focus on forecasting years with the greatest mortality. Identify the peak year for each agent.

In [ ]:
target_vars = list(filter(lambda x: x.endswith("_target"), westmort.variables.keys()))

idx_of_max_mort = westmort[target_vars].sum(dim=["x", "y"]).argmax(dim="time").to_dataarray()
year_of_max_mort = westmort.time.isel(time=idx_of_max_mort)
year_of_max_mort

## Generating data frames for training

Going from xarray -> dataframe will eat a lot of memory. One way to mitigate this is to call `westmort_merge.sel(...)` to get the years/insects you want and then convert to dataframe.

In [ ]:
COVARIATES = {
    "hydro": ["HT", "P50", "WUE", "rdmax", "gsmax"],
    "topo": ["elev", "heat"],
    "climate": ["tmin", "vpd", "def"]
}

In [ ]:
def make_data_frame(agent: str, valid_year: int, lookback_years: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    agent_ba = f"{agent}_ba"
    agent_mort = f"{agent}_mort"
    agent_target = f"{agent}_target"
    
    cols_to_select = COVARIATES["hydro"] +\
        COVARIATES["topo"] +\
        COVARIATES["climate"] +\
        [agent_ba, agent_mort, agent_target]

    start_year = valid_year - lookback_years

    # Apply a logit transform to prior-year mortality
    westmort_subset = westmort.sel(time=slice(f"{start_year}-01-01", f"{valid_year}-01-01"))
    westmort_subset[agent_mort] = gbm.safe_logit(westmort_subset[agent_mort])

    # Split into train/validation periods
    train = westmort_subset.isel(time=slice(None, -1)).to_dataframe()
    valid = westmort_subset.isel(time=[-1]).to_dataframe()

    # Only keep pixels with no nans and nonzero host ba
    train = train[train[agent_ba] > 0].dropna()
    valid = valid[valid[agent_ba] > 0].dropna()

    # Resample training data to have at most 50% pixels with zero mort
    train = gbm.balance_zeros(train, agent_target, nz_ratio=1.0)
    
    return train, valid

In [ ]:
%%time
train, valid = make_data_frame("fir_eng", 2010, 4)

In [ ]:
train.index.get_level_values("time").value_counts()

In [ ]:
valid.index.get_level_values("time").value_counts()

In [ ]:
(train["fir_eng_target"] > 0).value_counts()

## Model training functions

Here we are replicating the approach in Francis et al. (2025), but with a GBM instead of a simple logistic model. The full structure is:
 - Logit-transform response with bounds from 0-1
 - Hurdle model, with classifier arm doing quantile regression for the median
 - Invlogit-transform predictions

Logit transformation results in +/- Inf if we have zeros or ones in the input data. There are no ones, but we do have a lot of zeros. 

In [ ]:
def train_model(model: RegressorMixin, train: pd.DataFrame, valid: pd.DataFrame, target_var: str, valid_year: int) -> dict:
    if train.shape[0] == 0 or valid.shape[0] == 0:
        # Subsetting resulted in no data
        return {
            "year": start_year,
            "agent": target_var
        }
    
    train_x, train_y = gbm.split_xy(train, target_var)
    valid_x, valid_y = gbm.split_xy(valid, target_var)

    model.fit(train_x, train_y)

    valid_y_hat = model.predict(valid_x)
    train_y_hat = model.predict(train_x)

    valid_results = gbm.get_results(valid_y, valid_y_hat)
    train_results = gbm.get_results(train_y, train_y_hat)

    results = {}
    results["year"] = valid_year
    results["agent"] = target_var
    results["valid"] = valid_results
    results["train"] = train_results
    results["model"] = model

    return results, valid_y_hat

In [ ]:
warnings.filterwarnings("ignore", message="X does not have valid feature names")
logging.getLogger().setLevel(logging.CRITICAL)

## Parameter sensitivity analysis
The defaults were pretty reasonable for all of these so this code is not run. However, the code is here in case you want to try.
 - Lookback period
 - Number of estimators
 - Tree depth
 - Learning rate

## Temporal CV

Use a sliding 5-year window to derive training/validation sets.

In [ ]:
gbm_args = {
    "learning_rate": 0.1,
    "n_estimators": 100,
    "num_leaves": 16,
}

lookback_window_size = 4

year_min = 2001 # early surveys are wacky, ignore them
year_max = westmort.time.max().dt.year.data

valid_years = np.arange(year_min + lookback_window_size, year_max+1)
agents = list(const.HOST_DCA_CODES.keys())

param_space = list(product(valid_years, agents))
print("N iterations:", len(param_space))
print(param_space[0])

In [ ]:
temporal_cv_results = []
temporal_cv_predictions = []

for (valid_year, agent) in tqdm(param_space):
    target_var = f"{agent}_target"
    
    train, valid = make_data_frame(agent, valid_year, lookback_window_size)
    if train.shape[0] == 0 or valid.shape[0] == 0:
        continue # ignore years with empty data
        
    model = gbm.make_zif_quantile_estimator(gbm_args, gbm_args) # same args for classifier and regressor

    metrics, y_hat = train_model(
        model, train, valid, target_var, valid_year
    )

    temporal_cv_results.append(metrics)

    preds_df = pd.DataFrame(
        data=dict(
            agent=agent,
            y_true=valid[target_var],
            y_hat=y_hat
        ),
        index=valid.index
    )
    temporal_cv_predictions.append(preds_df)

temporal_cv_df = pd.json_normalize(temporal_cv_results)
temporal_cv_df.to_csv("../data_working/gbm_temporal_cv.csv")

In [ ]:
all_predictions = pd.concat(temporal_cv_predictions, axis=0)
all_predictions.to_parquet("../data_working/gbm_temporal_cv_predictions.parquet")

In [ ]:
fig, ax = plt.subplots()

temporal_cv_df.boxplot("valid.auc", by="agent", vert=False, ax=ax)
ax.set_xlim(-0.1, 1)
plt.show()